<a href="https://colab.research.google.com/github/Agouillal-Mayes/object-recognition-neural-network/blob/main/object_recogn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Téléverser le fichier kaggle.json
from google.colab import files
uploaded = files.upload()

# 2. Déplacer kaggle.json dans le dossier d'API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Télécharger le dataset avec le bon identifiant (ambityga/imagenet100)
!kaggle datasets download -d ambityga/imagenet100 --unzip -p /content/imagenet100

print("Téléchargement terminé !")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/ambityga/imagenet100
License(s): unknown
100% 16.1G/16.1G [02:33<00:00, 113MB/s]

Téléchargement terminé !


In [5]:
import os

base_path = '/content/imagenet100'

print("--- Contenu du dossier principal ---")
if os.path.exists(base_path):
    items = os.listdir(base_path)
    print(f"Le dossier '{base_path}' contient : {items}")

    for item in items:
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            sub_items = os.listdir(item_path)
            print(f"\nLe sous-dossier '{item}' contient {len(sub_items)} éléments (classes/fichiers).")
            if len(sub_items) > 0:
                print(f"Exemple d'éléments dans '{item}': {sub_items[:5]}")
else:
    print(f"Le dossier {base_path} n'existe pas.")

--- Contenu du dossier principal ---
Le dossier '/content/imagenet100' contient : ['train.X4', 'train.X1', 'val.X', 'Labels.json', 'train.X2', 'train.X3']

Le sous-dossier 'train.X4' contient 25 éléments (classes/fichiers).
Exemple d'éléments dans 'train.X4': ['n01667778', 'n01985128', 'n01630670', 'n01877812', 'n01496331']

Le sous-dossier 'train.X1' contient 25 éléments (classes/fichiers).
Exemple d'éléments dans 'train.X1': ['n01930112', 'n01665541', 'n01795545', 'n01775062', 'n01984695']

Le sous-dossier 'val.X' contient 100 éléments (classes/fichiers).
Exemple d'éléments dans 'val.X': ['n01667778', 'n01930112', 'n01685808', 'n01985128', 'n01665541']

Le sous-dossier 'train.X2' contient 25 éléments (classes/fichiers).
Exemple d'éléments dans 'train.X2': ['n01770081', 'n01740131', 'n01729977', 'n01774384', 'n01514668']

Le sous-dossier 'train.X3' contient 25 éléments (classes/fichiers).
Exemple d'éléments dans 'train.X3': ['n01685808', 'n01828970', 'n01614925', 'n01829413', 'n018470

In [6]:
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, ConcatDataset

# 1. Définition des transformations d'images
transform_train = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Chargement des datasets
# Les données d'entraînement sont réparties sur 4 dossiers (train.X1 à train.X4)
train_subsets = []
for i in range(1, 5):
    subset_path = f'/content/imagenet100/train.X{i}'
    train_subsets.append(ImageFolder(root=subset_path, transform=transform_train))

# Fusion des 4 sous-ensembles en un seul dataset d'entraînement
train_dataset = ConcatDataset(train_subsets)

# Chargement du dossier de validation
val_dataset = ImageFolder(root='/content/imagenet100/val.X', transform=transform_val)

# 3. Création des DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f" Images d'entraînement : {len(train_dataset)}")
print(f" Images de validation : {len(val_dataset)}")
# On récupère le nombre total de classes à partir de la validation (qui contient les 100 classes)
print(f" Nombre de classes : {len(val_dataset.classes)}")

 Images d'entraînement : 130000
 Images de validation : 5000
 Nombre de classes : 100
